In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

import os
import pathlib
from colorsys import hls_to_rgb, rgb_to_hls
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import seaborn as sns
from numpy.lib.stride_tricks import as_strided

import aeon
import datajoint as dj
from aeon.schema.schemas import social02
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from aeon.analysis.block_plotting import (
    subject_colors,
    subject_colors_dict,
    patch_markers,
    patch_markers_linestyles,
    gen_hex_grad,
)

In [ ]:
"""Set example block key, and print block patch names and subject names"""

key = {"experiment_name": "social0.2-aeon3", "block_start": "2024-02-11 13:36:10"}
patch_names, subject_names = (BlockSubjectAnalysis.Preference & key).fetch("patch_name", "subject_name")
patch_names = np.unique(patch_names)
subject_names = np.unique(subject_names)
print(patch_names, subject_names)

## Block Plots

In [ ]:
block_start, block_end = (Block & key).fetch1("block_start", "block_end")

### 1. Patch stats: mean, next to boxplots of each pellet threshold per patch.

In [ ]:
"""Fetch subject patch info."""

subj_patch_info = (BlockSubjectAnalysis.Patch & key).fetch(format="frame")
patch_info = (BlockAnalysis.Patch & key).fetch("patch_name", "patch_rate", "patch_offset", as_dict=True)
display(subj_patch_info)
display(patch_info)
subjects = subj_patch_info.index.get_level_values("subject_name").unique()

In [ ]:
"""Convert `subj_patch_info` into a form amenable to plotting."""

reset_subj_patch_info = subj_patch_info.reset_index()  # reset to turn MultiIndex into columns
min_subj_patch_info = reset_subj_patch_info[  # select only relevant columns
    ["patch_name", "subject_name", "pellet_timestamps", "patch_threshold"]
]
min_subj_patch_info = min_subj_patch_info.explode(
    ["pellet_timestamps", "patch_threshold"], ignore_index=True
).dropna().reset_index(drop=True)
# Rename and reindex columns
min_subj_patch_info.columns = ["patch", "subject", "time", "threshold"]
min_subj_patch_info = min_subj_patch_info.reindex(columns=["time", "patch", "threshold", "subject"])

display(min_subj_patch_info)

In [ ]:
"""Add patch mean values and block-normalized delivery times to pellet info."""

n_patches = len(patch_info)
patch_mean_info = pd.DataFrame(index=np.arange(n_patches), columns=min_subj_patch_info.columns)
patch_mean_info["subject"] = "mean"
patch_mean_info["patch"] = [d["patch_name"] for d in patch_info]
patch_mean_info["threshold"] = [((1 / d["patch_rate"]) + d["patch_offset"]) for d in patch_info]
patch_mean_info["time"] = subj_patch_info.index.get_level_values("block_start")[0]

min_subj_patch_info_plus = pd.concat((patch_mean_info, min_subj_patch_info)).reset_index(drop=True)
min_subj_patch_info_plus["norm_time"] = (
    (min_subj_patch_info_plus["time"] - min_subj_patch_info_plus["time"].iloc[0])
    / (min_subj_patch_info_plus["time"].iloc[-1] - min_subj_patch_info_plus["time"].iloc[0])
).round(3)

display(min_subj_patch_info_plus)

In [ ]:
"""Plot."""

box_colors = ["#0A0A0A"] + subject_colors[0 : len(subjects)]  # subject colors + mean color

fig = px.box(
    min_subj_patch_info_plus.sort_values("patch"),
    x="patch",
    y="threshold",
    color="subject",
    hover_data=["norm_time"],
    color_discrete_sequence=box_colors,
    # notched=True,
    points="all",
)

fig.update_layout(
    title="Patch Stats: Patch Means and Sampled Threshold Values",
    xaxis_title="Patch",
    yaxis_title="Threshold (cm)",
)

fig.show()

### 2. Animal weight: over time, per subject

In [ ]:
"""View Environment.SubjectWeight table."""

acquisition.Environment.SubjectWeight & key

In [ ]:
"""Get subject weights in block."""

chunk_restriction = (
    acquisition.create_chunk_restriction(key["experiment_name"], block_start, block_end)
)
weight_query = acquisition.Environment.SubjectWeight & key & chunk_restriction
weights = fetch_stream(weight_query)[block_start:block_end]
display(weight_query)
display(weights)

In [ ]:
"""Plot."""

fig = px.line(
    weights,
    x=weights.index,
    y="weight",
    color="subject_id",
    color_discrete_map=subject_colors_dict,
    markers=True,
)

fig.update_traces(line=dict(width=3), marker=dict(size=8))


fig.update_layout(
    title="Weights",
    xaxis_title="Time",
    yaxis_title="Weight (g)",
)

fig.show()